In [43]:
%%writefile app.py
# ============================================================
# ResumeFit AI v4.3 — Groq Default (Free) + Optional OpenAI
# Author: Bernard Lokasasmita
# Bootcamp: Ruangguru AI Engineering Batch 11
# ============================================================

import streamlit as st
import pypdf
import pandas as pd
import numpy as np
import re
import io
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import plotly.graph_objects as go
import plotly.express as px
from openai import OpenAI
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH

# PAGE CONFIG
st.set_page_config(
    page_title="ResumeFit AI v4.3",
    page_icon="🚀",
    layout="wide",
    initial_sidebar_state="expanded"
)

# CUSTOM CSS
st.markdown("""
<style>
    .main { background-color: #f8fafc; }
    .stButton>button {
        background: linear-gradient(135deg, #4f0d8f, #7c3aed);
        color: white; border: none; border-radius: 8px;
        padding: 12px 28px; font-size: 15px; font-weight: 700;
        width: 100%; cursor: pointer;
    }
    .stButton>button:hover { opacity: 0.9; }
    .header-box {
        background: linear-gradient(135deg, #4f0d8f, #7c3aed);
        padding: 28px 32px; border-radius: 12px; margin-bottom: 24px;
    }
    .aspect-card {
        background: white; border-radius: 10px; padding: 16px 20px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.07); margin-bottom: 12px;
        border-left: 5px solid #7c3aed;
    }
    .aspect-card h4 { margin: 0 0 6px 0; color: #4f0d8f; font-size: 1rem; }
    .aspect-card p  { margin: 0; color: #374151; font-size: 0.88rem; }
    .score-badge-green  { background:#dcfce7; color:#166534; border-radius:20px; padding:3px 12px; font-size:0.82rem; font-weight:700; }
    .score-badge-yellow { background:#fef9c3; color:#854d0e; border-radius:20px; padding:3px 12px; font-size:0.82rem; font-weight:700; }
    .score-badge-red    { background:#fee2e2; color:#991b1b; border-radius:20px; padding:3px 12px; font-size:0.82rem; font-weight:700; }
    .skill-found   { display:inline-block; background:#dcfce7; color:#166534; border-radius:20px; padding:4px 12px; margin:3px; font-size:0.83rem; font-weight:600; }
    .skill-missing { display:inline-block; background:#fee2e2; color:#991b1b; border-radius:20px; padding:4px 12px; margin:3px; font-size:0.83rem; font-weight:600; }
    .before-box { background:#fff7ed; border-left:4px solid #f97316; padding:12px 16px; border-radius:6px; font-size:0.88rem; color:#374151; white-space:pre-wrap; }
    .after-box  { background:#f0fdf4; border-left:4px solid #22c55e; padding:12px 16px; border-radius:6px; font-size:0.88rem; color:#374151; white-space:pre-wrap; }
    .section-header {
        background: linear-gradient(135deg, #4f0d8f, #7c3aed);
        color: white; padding: 10px 18px; border-radius: 8px;
        font-size: 1.05rem; font-weight: 700; margin: 18px 0 10px 0;
    }
    .metric-card {
        background: white; border-radius: 12px; padding: 18px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); text-align: center;
        border-left: 5px solid #7c3aed;
    }
    .metric-card h1 { font-size: 2.2rem; color: #4f0d8f; margin: 0; }
    .metric-card p  { color: #6b7280; margin: 4px 0 0 0; font-size: 0.85rem; }
    .error-box   { background:#fee2e2; color:#991b1b; padding:14px 18px; border-radius:8px; font-size:0.88rem; margin-top:8px; border-left:4px solid #ef4444; }
    .success-box { background:#dcfce7; color:#166534; padding:14px 18px; border-radius:8px; font-size:0.88rem; margin-top:8px; border-left:4px solid #22c55e; }
    .info-box    { background:#eff6ff; color:#1e40af; padding:14px 18px; border-radius:8px; font-size:0.88rem; margin-top:8px; border-left:4px solid #3b82f6; }
    .warn-box    { background:#fef9c3; color:#854d0e; padding:14px 18px; border-radius:8px; font-size:0.88rem; margin-top:8px; border-left:4px solid #f59e0b; }
    .rule-box    { background:#1e1e2e; color:#a6e3a1; padding:14px 18px; border-radius:8px; font-family:monospace; font-size:0.82rem; white-space:pre-wrap; margin-top:8px; }
    .strategy-box { background:#f5f3ff; border-left:4px solid #7c3aed; padding:14px 18px; border-radius:8px; font-size:0.9rem; color:#4f0d8f; margin:12px 0; }
</style>
""", unsafe_allow_html=True)

# ── CONSTANTS ──────────────────────────────────────────────
SKILL_DICT = [
    "python","java","javascript","typescript","c++","c#","r","scala","go","rust",
    "kotlin","swift","php","ruby","matlab","machine learning","deep learning","nlp",
    "computer vision","tensorflow","pytorch","keras","scikit-learn","pandas","numpy",
    "hugging face","transformers","langchain","openai","llm","rag","vector database",
    "pinecone","faiss","embeddings","sql","mysql","postgresql","mongodb","redis",
    "elasticsearch","spark","hadoop","airflow","dbt","kafka","aws","gcp","azure",
    "docker","kubernetes","ci/cd","git","github","terraform","linux","bash",
    "tableau","power bi","streamlit","plotly","matplotlib","seaborn",
    "communication","leadership","teamwork","problem solving","project management",
    "agile","scrum","excel","powerpoint","word","google sheets","jira","confluence",
    "prompt engineering","fine-tuning","model deployment","api","fastapi","flask",
    "django","n8n","automation","opencv","yolo","product management","product roadmap",
    "stakeholder management","go-to-market","brand strategy","market research",
    "competitor analysis","sales strategy","product launch","pricing strategy"
]

ASPECTS = [
    ("Overall Impression",          "🌟", "General quality and first impression of the CV"),
    ("Contact Information",         "📞", "Completeness of contact details"),
    ("Relevant Skills",             "🛠️",  "Technical and soft skills alignment"),
    ("Professional Summary",        "📝", "Quality of the professional summary/objective"),
    ("Work Experience",             "💼", "Relevance and impact of work experience"),
    ("Achievements",                "🏆", "Quantified accomplishments and impact"),
    ("Education & Certification",   "🎓", "Educational background and certifications"),
    ("Organizational Activity",     "🤝", "Extracurricular and organizational involvement"),
    ("Consistent & Error-free",     "✅", "Grammar, formatting, and consistency"),
    ("Additional Sections",         "➕", "Portfolio, projects, languages, hobbies"),
    ("Keywords",                    "🔑", "ATS keyword optimization"),
    ("Career Recommendation",       "🎯", "Suggested career paths and roles"),
]

GROQ_BASE_URL      = "https://api.groq.com/openai/v1"
GROQ_DEFAULT_MODEL = "llama-3.1-8b-instant"
OPENAI_MODELS      = ["gpt-3.5-turbo", "gpt-4o-mini", "gpt-4o"]

# ── LLM CLIENTS ────────────────────────────────────────────
def get_groq_client(api_key):
    return OpenAI(api_key=api_key, base_url=GROQ_BASE_URL)

def get_openai_client(api_key):
    return OpenAI(api_key=api_key)

# ── HELPERS ────────────────────────────────────────────────
def extract_text_from_pdf(file):
    try:
        reader = pypdf.PdfReader(file)
        text = ""
        for page in reader.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
        return text.strip()
    except Exception as e:
        st.error(f"PDF error: {e}")
        return ""

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s+#]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def extract_skills(text):
    tl = text.lower()
    return list({s for s in SKILL_DICT if s.lower() in tl})

def calculate_tfidf(resume, job):
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
    try:
        m = vec.fit_transform([clean_text(resume), clean_text(job)])
        return round(cosine_similarity(m[0:1], m[1:2])[0][0] * 100, 1)
    except:
        return 0.0

def keyword_coverage(resume, job):
    jw = {w for w in clean_text(job).split() if len(w) > 3}
    rw = set(clean_text(resume).split())
    if not jw:
        return 0.0, [], []
    matched = jw & rw
    missing = jw - rw
    return round(len(matched)/len(jw)*100, 1), list(matched)[:15], list(missing)[:15]

def skill_analysis(resume, job):
    rs = extract_skills(resume)
    js = extract_skills(job)
    return rs, js, list(set(rs)&set(js)), list(set(js)-set(rs)), list(set(rs)-set(js))

def score_label(s):
    if s >= 75: return "Excellent Match", "#22c55e"
    if s >= 55: return "Good Match",      "#3b82f6"
    if s >= 35: return "Moderate Match",  "#f59e0b"
    return "Needs Improvement", "#ef4444"

def badge(score):
    if score >= 75: return "<span class='score-badge-green'>Strong ✅</span>"
    if score >= 50: return "<span class='score-badge-yellow'>Average ⚠️</span>"
    return "<span class='score-badge-red'>Weak ❌</span>"

def clean_ai_text(text):
    text = re.sub(r"\*{1,3}(.*?)\*{1,3}", r"\1", text)
    text = re.sub(r"`{1,3}(.*?)`{1,3}", r"\1", text)
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    return text.strip()

# ── CHARTS ─────────────────────────────────────────────────
def gauge_chart(score):
    label, color = score_label(score)
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=score,
        title={"text": f"ATS Match Score<br><span style='font-size:0.8em;color:{color}'>{label}</span>", "font":{"size":15}},
        gauge={
            "axis": {"range":[0,100]},
            "bar":  {"color": color},
            "steps":[
                {"range":[0,35],  "color":"#fee2e2"},
                {"range":[35,55], "color":"#fef9c3"},
                {"range":[55,75], "color":"#dbeafe"},
                {"range":[75,100],"color":"#dcfce7"},
            ],
        }
    ))
    fig.update_layout(height=260, margin=dict(t=60,b=10,l=10,r=10))
    return fig

def radar_chart(matched, missing):
    all_s = (matched+missing)[:8]
    if len(all_s) < 3: return None
    rv = [1 if s in matched else 0 for s in all_s]
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=[1]*len(all_s)+[1], theta=all_s+[all_s[0]], fill="toself", name="Job Req", fillcolor="rgba(124,58,237,0.15)", line=dict(color="#7c3aed",width=2)))
    fig.add_trace(go.Scatterpolar(r=rv+[rv[0]], theta=all_s+[all_s[0]], fill="toself", name="Your CV", fillcolor="rgba(34,197,94,0.2)", line=dict(color="#22c55e",width=2)))
    fig.update_layout(polar=dict(radialaxis=dict(visible=True,range=[0,1])), showlegend=True, height=300, margin=dict(t=40,b=30,l=30,r=30), title="Skill Radar")
    return fig

def aspect_bar_chart(scores_dict):
    aspects = list(scores_dict.keys())
    scores  = [v if isinstance(v, (int,float)) else v.get("score",0) for v in scores_dict.values()]
    colors  = ["#22c55e" if s>=75 else "#f59e0b" if s>=50 else "#ef4444" for s in scores]
    fig = go.Figure(go.Bar(x=scores, y=aspects, orientation='h', marker_color=colors, text=[f"{s}%" for s in scores], textposition="auto"))
    fig.update_layout(height=420, margin=dict(t=20,b=20,l=10,r=10), xaxis=dict(range=[0,100]), title="12-Aspect CV Score Breakdown")
    return fig

# ── AI FUNCTIONS ───────────────────────────────────────────
def test_api_connection(client, model):
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say OK"}],
            max_tokens=5
        )
        _ = r.choices[0].message.content
        return True, f"Connected ✅  (model: {model})"
    except Exception as e:
        return False, str(e)

def analyze_12_aspects(resume_text, job_text, client, model):
    system_msg = (
        "You are a Strategic Career Architect. Provide a brutally honest, specific gap analysis "
        "between this candidate and the job. Be concrete — reference actual content from the resume. "
        "No markdown, no bold, no bullet points. Plain text only."
    )
    user_msg = (
        f"RESUME:\n{resume_text[:2000]}\n\n"
        f"JOB DESCRIPTION:\n{job_text[:1200]}\n\n"
        "Analyze ALL 12 aspects. For each, output EXACTLY this block:\n"
        "ASPECT: <name>\n"
        "SCORE: <0-100>\n"
        "ASSESSMENT: <one specific sentence referencing the resume>\n"
        "TIP: <one concrete actionable improvement>\n"
        "---\n\n"
        "ASPECTS (all 12 in order):\n"
        "1. Overall Impression\n2. Contact Information\n3. Relevant Skills\n"
        "4. Professional Summary\n5. Work Experience\n6. Achievements\n"
        "7. Education & Certification\n8. Organizational Activity\n"
        "9. Consistent & Error-free Writing\n10. Additional Sections\n"
        "11. Keywords\n12. Career Recommendation\n\n"
        "RULES: Plain text only. No **, no *, no #. All 12 blocks required. No intro or closing text."
    )
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
            max_tokens=2800,
            temperature=0.2
        )
        return r.choices[0].message.content, None
    except Exception as e:
        return "", str(e)

def parse_aspect_scores(raw_text):
    results = {}
    if not raw_text:
        return results
    blocks = re.split(r"\n\s*-{2,}\s*\n|\n\s*-{2,}\s*$|^\s*-{2,}\s*$", raw_text, flags=re.MULTILINE)
    for block in blocks:
        if not block.strip():
            continue
        name = score = assessment = tip = None
        for line in block.split("\n"):
            l = line.strip().replace("**","").replace("`","")
            m = re.match(r"(?i)^aspect\s*[:\-]\s*(.+)$", l)
            if m: name = m.group(1).strip()
            m = re.match(r"(?i)^score\s*[:\-]\s*([0-9]{1,3})", l)
            if m: score = m.group(1).strip()
            m = re.match(r"(?i)^assessment\s*[:\-]\s*(.+)$", l)
            if m: assessment = m.group(1).strip()
            m = re.match(r"(?i)^tip\s*[:\-]\s*(.+)$", l)
            if m: tip = m.group(1).strip()
        if name and score:
            try:
                s = max(0, min(100, int(re.sub(r'[^0-9]', '', score))))
                results[name] = {"score": s, "assessment": assessment or "", "tip": tip or ""}
            except:
                pass
    return results

def rewrite_cv_sections(resume_text, job_text, missing_skills, final_score, client, model):
    """
    Smart dual-mode rewrite:
    - HIGH match (>=60%): tailor CV to job
    - LOW match (<60%):  enhance CV quality + suggest pivot, do NOT force-fit
    """
    ms = ', '.join(missing_skills[:10]) if missing_skills else 'None'

    if final_score >= 60:
        strategy = "HIGH_MATCH"
        strategy_instruction = (
            f"The candidate has a {final_score}% match with this job. "
            "STRATEGY: Tailor the CV to this specific job. "
            "Align wording with the job description. Emphasize matching skills. "
            "Incorporate missing keywords naturally where truthful."
        )
    else:
        strategy = "LOW_MATCH"
        strategy_instruction = (
            f"The candidate has only a {final_score}% match with this job — the CV is NOT a strong fit. "
            "STRATEGY: Do NOT force-fit the candidate into this role. "
            "Instead, ENHANCE the CV to be stronger overall: "
            "improve clarity, sharpen impact statements, highlight transferable skills, "
            "and suggest realistic pivot roles. Preserve the candidate's authentic identity."
        )

    system_msg = (
        "You are a senior career coach and elite resume strategist. "
        "Your job is to improve the candidate's CV while remaining 100% truthful. "
        "Do NOT fabricate experience, skills, or metrics. "
        "Do NOT change the candidate's career identity. "
        "Plain text only — no **, no *, no #, no backticks. ALL CAPS for section headers only."
    )

    user_msg = (
        f"RESUME:\n{resume_text[:2000]}\n\n"
        f"JOB DESCRIPTION:\n{job_text[:1200]}\n\n"
        f"MATCH SCORE: {final_score}%\n"
        f"MISSING SKILLS: {ms}\n\n"
        f"COACHING STRATEGY: {strategy_instruction}\n\n"
        "OUTPUT ALL 5 SECTIONS in this exact format (plain text, no markdown):\n\n"
        "PROFESSIONAL SUMMARY\n"
        "[3-4 powerful sentences. Start with a strong title. No 'I am' or 'I seek'. "
        "Focus on value delivered, not duties. Match strategy above.]\n\n"
        "TOP 5 ACHIEVEMENT BULLETS\n"
        "- [Accomplished X as measured by Y by doing Z — improve existing bullets, do NOT invent metrics]\n"
        "- [Leadership or scale impact]\n"
        "- [Tool or technical application]\n"
        "- [Process improvement or collaboration]\n"
        "- [Business growth or customer outcome]\n\n"
        "SKILLS SECTION\n"
        "[Top 15 most relevant skills for this candidate's actual background, comma-separated]\n\n"
        "KEYWORDS TO ADD\n"
        "[8 high-value keywords the candidate should add — only if truthfully applicable]\n\n"
        "CAREER RECOMMENDATION\n"
        "[Honest coaching: best-fit roles for this candidate, one specific skill/cert to pursue next, "
        "and if low match — which roles are a better fit than this job]\n\n"
        "RULES:\n"
        "- Do NOT use 'I' or 'My'. Use third-person action verbs.\n"
        "- Do NOT fabricate numbers, companies, or skills.\n"
        "- Plain text only. No markdown.\n"
        "- Output all 5 sections. No intro or closing text.\n"
    )

    try:
        r = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user",   "content": user_msg}
            ],
            max_tokens=2200,
            temperature=0.4
        )
        return r.choices[0].message.content, None, strategy
    except Exception as e:
        return "", str(e), strategy

# ── RULE-BASED FALLBACK ────────────────────────────────────
def rule_based_aspect_scores(resume_text, job_text, final_score, skill_score, cov_pct):
    scores = {}
    rt = resume_text.lower()
    has_email    = "@" in rt
    has_phone    = bool(re.search(r'\d{3}[-.\s]\d{3,4}', rt))
    has_linkedin = "linkedin" in rt
    has_summary  = any(w in rt for w in ["summary","objective","profile","about"])
    has_exp      = any(w in rt for w in ["experience","work","employment","position"])
    has_edu      = any(w in rt for w in ["education","university","degree","bachelor","master"])
    has_achieve  = any(w in rt for w in ["achieved","increased","reduced","improved","led","managed"])
    has_numbers  = bool(re.search(r'\d+%|\d+ years|\$\d+', rt))
    has_org      = any(w in rt for w in ["organization","club","volunteer","committee","association"])
    has_extra    = any(w in rt for w in ["portfolio","github","project","certification","language"])
    word_count   = len(resume_text.split())
    scores["Overall Impression"]        = min(100, int(final_score * 1.1))
    scores["Contact Information"]       = 60 + (10 if has_email else 0) + (10 if has_phone else 0) + (10 if has_linkedin else 0)
    scores["Relevant Skills"]           = int(skill_score)
    scores["Professional Summary"]      = 70 if has_summary else 35
    scores["Work Experience"]           = 75 if has_exp else 30
    scores["Achievements"]              = (70 if has_achieve else 35) + (10 if has_numbers else 0)
    scores["Education & Certification"] = 75 if has_edu else 40
    scores["Organizational Activity"]   = 70 if has_org else 35
    scores["Consistent & Error-free"]   = 65 if word_count > 200 else 45
    scores["Additional Sections"]       = 70 if has_extra else 40
    scores["Keywords"]                  = int(cov_pct)
    scores["Career Recommendation"]     = int(final_score * 0.9)
    return {k: min(v, 100) for k, v in scores.items()}

def rule_based_rewrite(resume_text, job_text, missing_skills, matched_skills, final_score):
    strategy_note = (
        "✅ Strategy: Tailoring CV to job (match >= 60%)"
        if final_score >= 60 else
        "⚠️ Strategy: Enhancing CV quality (low match — not force-fitting to this job)"
    )
    out = [strategy_note, ""]
    out.append("### PROFESSIONAL SUMMARY")
    skills_str = ', '.join(matched_skills[:4]) if matched_skills else 'relevant fields'
    out.append(
        f"Results-driven professional with proven expertise in {skills_str}. "
        "Demonstrated ability to deliver measurable outcomes through strategic thinking and cross-functional collaboration. "
        "Committed to continuous improvement and driving business value."
    )
    out.append("")
    out.append("### TOP 5 ACHIEVEMENT BULLETS")
    for b in [
        "Accomplished [specific task] as measured by [metric/result] by doing [method/tool used].",
        "Led [project/initiative] resulting in [X]% improvement in [area] through [approach].",
        "Developed [solution/system] that reduced [problem] by [metric] using [technology].",
        "Collaborated with [team/stakeholders] to deliver [outcome] within [timeframe].",
        "Optimized [process/workflow] achieving [result] by implementing [tool/method].",
    ]:
        out.append(f"- {b}")
    out.append("")
    out.append("### SKILLS SECTION")
    all_skills = matched_skills + missing_skills[:5]
    out.append(', '.join(all_skills) if all_skills else "Add relevant technical and soft skills here.")
    out.append("")
    out.append("### KEYWORDS TO ADD")
    out.append(', '.join(missing_skills[:8]) if missing_skills else "Your CV already covers most keywords.")
    out.append("")
    out.append("### CAREER RECOMMENDATION")
    out.append(
        "Based on your skill profile, consider roles aligned with your strongest competencies. "
        "Focus on building portfolio projects that demonstrate end-to-end impact. "
        "Pursue one relevant certification to strengthen your positioning."
    )
    out.append("")
    out.append("---")
    out.append("⚠️ This is a rule-based suggestion. Enable AI for personalized LLM-powered rewrite.")
    return "\n".join(out)

# ── EXPORT ─────────────────────────────────────────────────
def export_docx(resume_text, rewritten_content, final_score, aspect_scores, matched_skills, missing_skills):
    doc = Document()
    title = doc.add_heading("ResumeFit AI — Optimized CV Report", 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_heading("ATS Match Score Summary", level=1)
    table = doc.add_table(rows=1, cols=2)
    table.style = 'Table Grid'
    hdr = table.rows[0].cells
    hdr[0].text = "Metric"; hdr[1].text = "Score"
    for m, v in [
        ("Overall ATS Match Score", f"{final_score}%"),
        ("Matched Skills", str(len(matched_skills))),
        ("Missing Skills", str(len(missing_skills))),
    ]:
        row = table.add_row().cells
        row[0].text = m; row[1].text = v
    doc.add_paragraph()
    if aspect_scores:
        doc.add_heading("12-Aspect CV Analysis", level=1)
        for asp, data in aspect_scores.items():
            p = doc.add_paragraph()
            p.add_run(f"{asp}: {data['score']}/100").bold = True
            doc.add_paragraph(f"  Assessment: {data.get('assessment','')}")
            doc.add_paragraph(f"  Tip: {data.get('tip','')}")
    doc.add_paragraph()
    doc.add_heading("Matched Skills", level=1)
    doc.add_paragraph(', '.join(matched_skills) if matched_skills else "None detected")
    doc.add_heading("Missing Skills", level=1)
    doc.add_paragraph(', '.join(missing_skills) if missing_skills else "None detected")
    doc.add_paragraph()
    if rewritten_content:
        doc.add_heading("AI-Optimized CV Sections", level=1)
        for line in rewritten_content.split("\n"):
            if line.strip():
                if any(line.strip().upper().startswith(x) for x in ["PROFESSIONAL","TOP 5","SKILLS","KEYWORDS","CAREER"]):
                    doc.add_paragraph(line.strip()).runs[0].bold = True
                else:
                    doc.add_paragraph(line.strip())
    doc.add_paragraph()
    doc.add_paragraph("Generated by ResumeFit AI v4.3 | Bernard Lokasasmita | AI Engineering Bootcamp Batch 11")
    buf = io.BytesIO()
    doc.save(buf)
    buf.seek(0)
    return buf

def export_txt(resume_text, rewritten_content, final_score, matched_skills, missing_skills, aspect_scores):
    out = [
        "=" * 60,
        "ResumeFit AI v4.3 — Optimized CV Report",
        "=" * 60,
        f"Overall ATS Match Score : {final_score}%",
        f"Matched Skills          : {len(matched_skills)}",
        f"Missing Skills          : {len(missing_skills)}",
        "",
        "--- 12-ASPECT ANALYSIS ---",
    ]
    if aspect_scores:
        for asp, data in aspect_scores.items():
            out.append(f"\n{asp}: {data['score']}/100")
            out.append(f"  {data.get('assessment','')}")
            out.append(f"  Tip: {data.get('tip','')}")
    out += [
        "",
        "--- MATCHED SKILLS ---",
        ', '.join(matched_skills) if matched_skills else "None",
        "",
        "--- MISSING SKILLS ---",
        ', '.join(missing_skills) if missing_skills else "None",
        "",
        "--- AI-OPTIMIZED SECTIONS ---",
        rewritten_content if rewritten_content else "No rewrite available.",
        "",
        "Generated by ResumeFit AI v4.3 | Bernard Lokasasmita | AI Engineering Bootcamp Batch 11"
    ]
    return "\n".join(out)

# ── SIDEBAR ────────────────────────────────────────────────
with st.sidebar:
    st.image("https://img.icons8.com/fluency/96/resume.png", width=70)
    st.title("ResumeFit AI")
    st.caption("v4.3 — Groq (Free) + Optional OpenAI")
    st.divider()
    st.markdown("### ⚙️ Settings")
    use_ai = st.toggle("Enable AI Analysis", value=True)
    llm_client = None
    llm_model  = GROQ_DEFAULT_MODEL
    use_openai = False

    if use_ai:
        st.markdown("**LLM Provider**")
        use_openai = st.toggle("Switch to ChatGPT (OpenAI)", value=False,
                               help="By default, Groq (free & fast) is used.")
        if not use_openai:
            groq_key = os.environ.get("GROQ_API_KEY", "").strip()
            if groq_key:
                llm_client = get_groq_client(groq_key)
                st.markdown("<div class='success-box'>🟢 <b>Groq ready</b> — Free &amp; Fast ⚡</div>", unsafe_allow_html=True)
            else:
                groq_key_input = st.text_input("Groq API Key (free)", type="password", placeholder="gsk_...",
                                               help="Get your FREE key at https://console.groq.com")
                st.caption("🔑 Free key: [console.groq.com](https://console.groq.com)")
                if groq_key_input:
                    llm_client = get_groq_client(groq_key_input)
                    st.markdown("<div class='success-box'>🟢 <b>Groq ready</b> — Free &amp; Fast ⚡</div>", unsafe_allow_html=True)
                else:
                    st.markdown(
                        "<div class='info-box'>ℹ️ Add <b>GROQ_API_KEY</b> to Colab Secrets for auto-login, "
                        "or paste it above.<br>Get a free key at "
                        "<a href='https://console.groq.com' target='_blank'>console.groq.com</a></div>",
                        unsafe_allow_html=True
                    )
        else:
            st.markdown("---")
            openai_key = st.text_input("OpenAI API Key", type="password", placeholder="sk-...")
            llm_model  = st.selectbox("OpenAI Model", OPENAI_MODELS, index=0)
            if openai_key:
                if st.button("🔌 Test Connection"):
                    with st.spinner("Testing..."):
                        tmp = get_openai_client(openai_key)
                        ok, msg = test_api_connection(tmp, llm_model)
                    if ok:
                        st.markdown(f"<div class='success-box'>{msg}</div>", unsafe_allow_html=True)
                        llm_client = get_openai_client(openai_key)
                    else:
                        st.markdown(f"<div class='error-box'>❌ {msg}</div>", unsafe_allow_html=True)
                else:
                    llm_client = get_openai_client(openai_key)
            else:
                st.markdown("<div class='info-box'>ℹ️ Enter your OpenAI API key above.</div>", unsafe_allow_html=True)
    else:
        st.info("💡 AI OFF — using rule-based engine.\nToggle AI above for LLM-powered analysis.")

    st.divider()
    st.markdown("### 📋 12 CV Aspects")
    for name, icon, _ in ASPECTS:
        st.markdown(f"{icon} {name}")
    st.divider()
    st.markdown("### 📊 Score Guide")
    st.markdown("""
- 🟢 **75–100%** Excellent
- 🔵 **55–75%** Good
- 🟡 **35–55%** Moderate
- 🔴 **0–35%** Needs Work
""")
    st.divider()
    st.caption("ResumeFit AI v4.3 | Bernard Lokasasmita | Batch 11")

# ── HEADER ─────────────────────────────────────────────────
st.markdown("""
<div class='header-box'>
    <h1 style='color:white;margin:0;font-size:2rem;'>🚀 ResumeFit AI <span style='font-size:1rem;opacity:0.8;'>v4.3</span></h1>
    <p style='color:#e9d5ff;margin:6px 0 0 0;font-size:1rem;'>
        Bedah 12 Aspek CV Kamu — ATS Matching, Skill Gap, Smart AI Rewrite &amp; Export
    </p>
</div>
""", unsafe_allow_html=True)

if use_ai and llm_client:
    provider = "OpenAI / ChatGPT" if use_openai else "Groq (Free ⚡)"
    st.markdown(
        f"<div class='success-box'>🤖 <b>AI Mode ON</b> — Provider: <b>{provider}</b> &nbsp;|&nbsp; Model: <b>{llm_model}</b></div>",
        unsafe_allow_html=True
    )
elif use_ai and not llm_client:
    st.markdown(
        "<div class='info-box'>⚠️ <b>AI Mode ON</b> — Enter your Groq API key in the sidebar to activate. "
        "(<a href='https://console.groq.com' target='_blank'>Get free key</a>)</div>",
        unsafe_allow_html=True
    )
else:
    st.markdown(
        "<div class='rule-box'>🔧 Rule-Based Mode — AI is OFF. Toggle AI in sidebar for LLM-powered analysis.</div>",
        unsafe_allow_html=True
    )
st.markdown("<br>", unsafe_allow_html=True)

# ── INPUT ──────────────────────────────────────────────────
st.markdown("<div class='section-header'>📄 Step 1: Upload CV &amp; Job Description</div>", unsafe_allow_html=True)
col1, col2 = st.columns(2)
with col1:
    st.markdown("**Upload CV / Resume (PDF)**")
    uploaded_file = st.file_uploader("", type=["pdf"], label_visibility="collapsed")
    if uploaded_file:
        st.success(f"✅ {uploaded_file.name}")
with col2:
    st.markdown("**Paste Job Description**")
    job_description = st.text_area("", height=200,
                                   placeholder="Paste the full job description here...",
                                   label_visibility="collapsed")
    if job_description:
        st.caption(f"📝 {len(job_description.split())} words")

st.markdown("<br>", unsafe_allow_html=True)
analyze_btn = st.button("🔍 Analyze My CV — 12 Aspects", use_container_width=True)

# ── ANALYSIS ───────────────────────────────────────────────
if analyze_btn:
    if not uploaded_file:
        st.error("⚠️ Please upload your CV PDF.")
        st.stop()
    if not job_description or len(job_description.strip()) < 50:
        st.error("⚠️ Please paste a job description (min 50 chars).")
        st.stop()
    if use_ai and not llm_client:
        st.error("⚠️ AI is enabled but no valid API key. Please enter your key in the sidebar.")
        st.stop()

    with st.spinner("🤖 Running analysis..."):
        resume_text = extract_text_from_pdf(uploaded_file)
        if not resume_text:
            st.error("Could not extract text. Use a text-based PDF.")
            st.stop()

        tfidf_score = calculate_tfidf(resume_text, job_description)
        cov_pct, matched_kw, missing_kw = keyword_coverage(resume_text, job_description)
        rs, js, matched_s, missing_s, extra_s = skill_analysis(resume_text, job_description)
        skill_score = (len(matched_s) / max(len(js), 1)) * 100
        final_score = round((tfidf_score*0.5) + (cov_pct*0.3) + (skill_score*0.2), 1)
        label, color = score_label(final_score)

        aspect_scores = {}
        rewritten_cv  = ""
        rewrite_strategy = ""
        ai_errors = []

        if use_ai and llm_client:
            provider_label = "OpenAI" if use_openai else "Groq"

            with st.spinner(f"🧠 Running 12-aspect analysis via {provider_label} ({llm_model})..."):
                raw_aspect, err1 = analyze_12_aspects(resume_text, job_description, llm_client, llm_model)
                if err1:
                    ai_errors.append(f"12-Aspect Analysis: {err1}")
                else:
                    aspect_scores = parse_aspect_scores(raw_aspect)
                    if len(aspect_scores) < 12:
                        rb = rule_based_aspect_scores(resume_text, job_description, final_score, skill_score, cov_pct)
                        for k, v in rb.items():
                            aspect_scores.setdefault(k, {
                                "score": v,
                                "assessment": "Rule-based estimate (AI response format varied).",
                                "tip": "Re-run analysis for AI-specific tips."
                            })

            with st.spinner(f"✍️ Rewriting CV sections via {provider_label} ({llm_model})..."):
                rewritten_cv, err2, rewrite_strategy = rewrite_cv_sections(
                    resume_text, job_description, missing_s, final_score, llm_client, llm_model
                )
                if err2:
                    ai_errors.append(f"CV Rewrite: {err2}")

            for err in ai_errors:
                st.markdown(
                    f"<div class='error-box'>❌ AI Error: {err}<br><br>"
                    "💡 <b>Fix:</b> Check your API key or try a different model.</div>",
                    unsafe_allow_html=True
                )
        else:
            rb_raw = rule_based_aspect_scores(resume_text, job_description, final_score, skill_score, cov_pct)
            aspect_scores = {
                k: {"score": v, "assessment": "Rule-based estimate.", "tip": "Enable AI for specific tips."}
                for k, v in rb_raw.items()
            }
            rewritten_cv = rule_based_rewrite(resume_text, job_description, missing_s, matched_s, final_score)
            rewrite_strategy = "HIGH_MATCH" if final_score >= 60 else "LOW_MATCH"

    st.success("✅ Analysis Complete!")
    st.divider()

    tab1, tab2, tab3, tab4 = st.tabs([
        "📊 Dashboard",
        "🔍 12-Aspect Analysis",
        "🤖 AI-Optimized CV",
        "📥 Export"
    ])

    # ── TAB 1: DASHBOARD ───────────────────────────────────
    with tab1:
        st.markdown("<div class='section-header'>📊 Your Results Dashboard</div>", unsafe_allow_html=True)
        r1, r2, r3 = st.columns([1.2, 1.2, 1])
        with r1:
            st.plotly_chart(gauge_chart(final_score), use_container_width=True)
        with r2:
            rc = radar_chart(matched_s, missing_s)
            if rc: st.plotly_chart(rc, use_container_width=True)
            else:  st.info("Not enough skills for radar chart.")
        with r3:
            st.markdown("<br>", unsafe_allow_html=True)
            for val, sub in [
                (f"{final_score}%", "Overall Score"),
                (f"{cov_pct}%",     "Keyword Coverage"),
                (f"{len(matched_s)}/{len(js)}", "Skills Matched"),
            ]:
                st.markdown(f"""
                <div class='metric-card'>
                    <h1>{val}</h1>
                    <p>{sub}</p>
                </div><br>
                """, unsafe_allow_html=True)

        st.divider()

        # Strategy Banner
        if final_score >= 60:
            st.markdown(
                f"<div class='success-box'>✅ <b>Strong Match ({final_score}%)</b> — "
                "AI will tailor your CV to this specific job.</div>",
                unsafe_allow_html=True
            )
        else:
            st.markdown(
                f"<div class='warn-box'>⚠️ <b>Low Match ({final_score}%)</b> — "
                "This job may not be the best fit. AI will enhance your CV quality overall "
                "and suggest better-aligned roles instead of force-fitting.</div>",
                unsafe_allow_html=True
            )

        st.markdown("**✅ Matched Skills**")
        if matched_s:
            st.markdown(" ".join([f"<span class='skill-found'>{s}</span>" for s in matched_s]), unsafe_allow_html=True)
        else:
            st.warning("No matching skills found.")
        st.markdown("<br>**❌ Missing Skills**", unsafe_allow_html=True)
        if missing_s:
            st.markdown(" ".join([f"<span class='skill-missing'>{s}</span>" for s in missing_s]), unsafe_allow_html=True)
        else:
            st.success("No critical skill gaps!")

        st.divider()
        m1, m2, m3, m4 = st.columns(4)
        m1.metric("Overall Score",    f"{final_score}%", delta=f"{final_score-50:.1f}% vs baseline")
        m2.metric("TF-IDF Score",     f"{tfidf_score}%")
        m3.metric("Keyword Coverage", f"{cov_pct}%")
        m4.metric("Skill Match",      f"{len(matched_s)}/{len(js)}")

    # ── TAB 2: 12-ASPECT ───────────────────────────────────
    with tab2:
        st.markdown("<div class='section-header'>🔍 Bedah 12 Aspek CV Kamu</div>", unsafe_allow_html=True)
        if aspect_scores:
            scores_dict = {k: v["score"] for k, v in aspect_scores.items()}
            st.plotly_chart(aspect_bar_chart(scores_dict), use_container_width=True)
            st.divider()
            mode_label = (
                "🤖 " + ("OpenAI" if use_openai else "Groq") + f" / {llm_model}"
                if (use_ai and llm_client and not ai_errors) else "🔧 Rule-Based"
            )
            st.caption(f"Analysis mode: {mode_label}")
            cols = st.columns(2)
            for i, (asp_name, data) in enumerate(aspect_scores.items()):
                icon = ASPECTS[i][1] if i < len(ASPECTS) else "📌"
                with cols[i % 2]:
                    sc = data["score"]
                    st.markdown(f"""
                    <div class='aspect-card'>
                        <h4>{icon} {asp_name} — {sc}/100 &nbsp; {badge(sc)}</h4>
                        <p>📋 {data.get('assessment','')}</p>
                        <p>💡 <b>Tip:</b> {data.get('tip','')}</p>
                    </div>
                    """, unsafe_allow_html=True)
        else:
            st.warning("No aspect scores available. Check AI errors above.")

    # ── TAB 3: AI REWRITE ──────────────────────────────────
    with tab3:
        st.markdown("<div class='section-header'>🤖 AI-Optimized CV Sections</div>", unsafe_allow_html=True)
        if rewritten_cv:
            mode_label = (
                "🤖 " + ("OpenAI" if use_openai else "Groq") + f" / {llm_model}"
                if (use_ai and llm_client and not ai_errors) else "🔧 Rule-Based Engine"
            )
            st.caption(f"Generated by: {mode_label}")

            # Strategy Banner
            if rewrite_strategy == "HIGH_MATCH":
                st.markdown(
                    f"<div class='success-box'>✅ <b>Tailored Mode</b> — Match score {final_score}% is strong. "
                    "CV has been tailored to this specific job.</div>",
                    unsafe_allow_html=True
                )
            else:
                st.markdown(
                    f"<div class='warn-box'>⚠️ <b>Enhancement Mode</b> — Match score {final_score}% is low. "
                    "CV has been improved overall (not force-fitted). "
                    "See Career Recommendation for better-fit roles.</div>",
                    unsafe_allow_html=True
                )

            st.divider()
            bc, ac = st.columns(2)
            with bc:
                st.markdown("**🔴 Original CV (excerpt)**")
                st.markdown(f"<div class='before-box'>{resume_text[:600]}...</div>", unsafe_allow_html=True)
            with ac:
                st.markdown("**🟢 Optimized Sections (preview)**")
                clean_preview = clean_ai_text(rewritten_cv[:600])
                st.markdown(f"<div class='after-box'>{clean_preview}...</div>", unsafe_allow_html=True)

            st.divider()
            st.markdown("### 📋 Full Optimized Sections")

            SECTION_HEADERS = [
                "PROFESSIONAL SUMMARY", "TOP 5 ACHIEVEMENT BULLETS",
                "SKILLS SECTION", "KEYWORDS TO ADD", "CAREER RECOMMENDATION"
            ]
            SECTION_ICONS = {
                "PROFESSIONAL SUMMARY":    "📝",
                "TOP 5 ACHIEVEMENT BULLETS": "🏆",
                "SKILLS SECTION":          "🛠️",
                "KEYWORDS TO ADD":         "🔑",
                "CAREER RECOMMENDATION":   "🎯",
            }

            clean_full = clean_ai_text(rewritten_cv)
            for line in clean_full.split("\n"):
                stripped = line.strip()
                if not stripped:
                    st.write("")
                elif stripped.upper() in SECTION_HEADERS:
                    icon = SECTION_ICONS.get(stripped.upper(), "🔹")
                    st.markdown(f"#### {icon} {stripped.title()}")
                    st.markdown("---")
                elif stripped.startswith("- ") or stripped.startswith("• "):
                    st.markdown(f"&nbsp;&nbsp;{stripped}")
                else:
                    st.write(stripped)
        else:
            st.warning("No rewrite available. Check AI errors above.")

    # ── TAB 4: EXPORT ──────────────────────────────────────
    with tab4:
        st.markdown("<div class='section-header'>📥 Export Your Optimized CV Report</div>", unsafe_allow_html=True)
        st.markdown("Download your full analysis and optimized sections.")
        e1, e2 = st.columns(2)
        with e1:
            st.markdown("### 📄 Export as DOCX (Word)")
            docx_buf = export_docx(resume_text, rewritten_cv, final_score, aspect_scores, matched_s, missing_s)
            st.download_button(
                label="📥 Download Optimized CV (.docx)",
                data=docx_buf,
                file_name="resumefit_optimized_cv.docx",
                mime="application/vnd.openxmlformats-officedocument.wordprocessingml.document",
                use_container_width=True
            )
        with e2:
            st.markdown("### 📋 Export as TXT")
            txt_report = export_txt(resume_text, rewritten_cv, final_score, matched_s, missing_s, aspect_scores)
            st.download_button(
                label="📥 Download Report (.txt)",
                data=txt_report,
                file_name="resumefit_report.txt",
                mime="text/plain",
                use_container_width=True
            )
        st.divider()
        m1, m2, m3, m4 = st.columns(4)
        m1.metric("Overall Score",    f"{final_score}%")
        m2.metric("Keyword Coverage", f"{cov_pct}%")
        m3.metric("Skills Matched",   f"{len(matched_s)}/{len(js)}")
        m4.metric("Aspects Analyzed", "12")


Overwriting app.py


In [44]:
import os
import time
from pyngrok import ngrok
from google.colab import userdata

# 1. Install dependencies (Quick check)
print("Checking dependencies...")
!pip -q install streamlit pypdf scikit-learn plotly openai pyngrok python-docx

# 2. Configure ngrok using Colab Secrets
try:
    token = userdata.get('NGROK_AUTH_TOKEN').strip()
    ngrok.set_auth_token(token)
    os.environ["NGROK_AUTH_TOKEN"] = token
    print("NGROK_AUTH_TOKEN loaded successfully ✅")
except Exception as e:
    print("❌ Error loading NGROK_AUTH_TOKEN.")
    raise e

# 3. Force-load GROQ_API_KEY into environment so Streamlit sees it
try:
    groq_key = userdata.get('GROQ_API_KEY').strip()
    os.environ['GROQ_API_KEY'] = groq_key
    print("GROQ_API_KEY injected into environment ✅")
except:
    print("⚠️ GROQ_API_KEY not found in Secrets. App will default to rule-based mode.")

# 4. Cleanup existing processes
ngrok.kill()
!pkill streamlit

# 5. Run the Streamlit app
APP_FILE = "app.py"
if os.path.exists(APP_FILE):
    print(f"Restarting Streamlit: {APP_FILE} ...")
    get_ipython().system_raw(f'streamlit run {APP_FILE} --server.port 8501 --server.address 0.0.0.0 &')

    # Wait for the server to spin up
    time.sleep(5)

    try:
        # Connect ngrok tunnel
        tunnel = ngrok.connect(8501, "http")
        print("\n" + "="*60)
        print("🚀 SUCCESS: Your ResumeFit AI app is refreshed!")
        print(f"🔗 Public URL: {tunnel.public_url}")
        print("="*60)
    except Exception as e:
        print(f"\n❌ Ngrok Tunnel Error: {e}")
else:
    print(f"\n❌ ERROR: '{APP_FILE}' not found.")

Checking dependencies...
NGROK_AUTH_TOKEN loaded successfully ✅
GROQ_API_KEY injected into environment ✅
Restarting Streamlit: app.py ...

🚀 SUCCESS: Your ResumeFit AI app is refreshed!
🔗 Public URL: https://radial-pruning-preoccupy.ngrok-free.dev
